# Question 4: ML + Quantum + Parallel Computing
## Optimal Exam Scheduling under Clash Constraints
---
**Student Name:** Siyam Bubere  
**Registration ID:** 241070012, TY B.Tech Computer Engineering (Semester 5)  
**Lab / Course:** Optimization Techniques Lab (OT-LABEXAM-Q4)  
**Seed:** `random_seed = 012` (derived from Reg ID `241070012`)  

---
**Course Outcomes Covered:**
- **CO1**: Machine Learning (Support Vector Regression for Clash Prediction)
- **CO2**: Quantum Computing & Optimization (QUBO Formulation & Simulated Annealing)
- **CO4**: Parallel High-Performance Computing (Multiprocessing Scaling & Acceleration)

### Problem Description
An exam cell must schedule 4 exams into 2 available time slots. Students enrolled in two exams cannot sit them concurrently. However, exact clash counts are not known beforehand and must be predicted from enrolment features (department, year, elective status, class size). The resulting combinatorial assignment problem is formulated as a Quadratic Unconstrained Binary Optimization (QUBO) problem with 8 binary variables and solved using Simulated Quantum Annealing, with parallel multiprocessing used to accelerate independent restarts.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_SEED, QUBO_PENALTY_LAMBDA, EXAMS
from src.ml_svr import generate_synthetic_enrolment_data, train_and_evaluate_svr, predict_clashes_for_exams
from src.qubo_builder import build_qubo_matrix, brute_force_solver, decode_solution
from src.quantum_solver import solve_qubo_quantum, compare_with_brute_force
from src.parallel_runner import run_parallel_restarts
from src.conclusion import generate_three_line_conclusion

print("Environment initialized. Academic Random Seed =", RANDOM_SEED)

## Part a: Machine Learning (Support Vector Regression) [01 Mark, CO1]
- Generate 600 synthetic enrolment rows with `random_seed = 012`.
- Train/Test split: 70/30 (420 train, 180 test).
- Train SVR with RBF kernel and standard scaling.
- Report test RMSE and R².
- Predict pairwise student clashes for each of the 4 exams.

In [ ]:
df, X, y = generate_synthetic_enrolment_data(seed=RANDOM_SEED)
model, metrics, X_train, X_test, y_test, y_pred = train_and_evaluate_svr(X, y, seed=RANDOM_SEED)
clash_matrix, pair_preds, exam_summary = predict_clashes_for_exams(model)

print(f"Generated Rows: {len(df)} | Train: {metrics['n_train']} | Test: {metrics['n_test']}")
print(f"SVR Test RMSE: {metrics['test_rmse']:.4f}")
print(f"SVR Test R²:   {metrics['test_r2']:.4f}")

print("\nPairwise Clash Predictions:")
for (e1, e2), pred in pair_preds.items():
    print(f"  Exam {e1} vs Exam {e2}: {pred:.2f} clashing students")

pd.DataFrame(clash_matrix, index=[e.name for e in EXAMS], columns=[e.name for e in EXAMS])

## Part b: QUBO Formulation & Brute Force Verification [01 Mark, CO2]
- Decision variables: $x_{i,s} \in \{0, 1\}$ for exam $i \in \{1..4\}$, slot $s \in \{1..2\}$ (8 total).
- Single-slot rule: $x_{i,1} + x_{i,2} = 1$.
- Penalty: $\lambda \sum_{i=1}^4 (x_{i,1} + x_{i,2} - 1)^2$.
- Objective: $\sum_{s=1}^2 \sum_{i < j} C_{ij} x_{i,s} x_{j,s}$.
- Brute force check across all $2^8 = 256$ states.

In [ ]:
Q, offset = build_qubo_matrix(clash_matrix, lambda_penalty=QUBO_PENALTY_LAMBDA)
bf_results = brute_force_solver(Q, offset, clash_matrix, lambda_penalty=QUBO_PENALTY_LAMBDA)

print("QUBO Q Matrix (8x8):")
print(np.round(Q, 2))
print(f"Offset: {offset:.2f}")
print(f"Total States: {bf_results['total_states_evaluated']} | Valid States: {bf_results['total_valid_states']}")
print(f"Ground State Energy: {bf_results['ground_energy']:.2f}")

for s in bf_results["best_states"]:
    print(f"Optimal State [{s['bitstring']}]: Clash Energy = {s['total_energy']:.2f} | Slot 1: {s['slot_1']} | Slot 2: {s['slot_2']}")

## Part c: Quantum Computing (Simulated Annealing on QUBO) [1.5 Marks, CO2]
- Solve the QUBO using simulated quantum annealing (1000 reads, 300 sweeps).
- Report best answer, energy, frequency, and compare against brute force ground truth.

In [ ]:
raw_samples, q_summary = solve_qubo_quantum(Q, offset, num_reads=1000, num_sweeps=300, seed=RANDOM_SEED)
comparison = compare_with_brute_force(q_summary, bf_results)

print(f"Best Bitstring Found: [{q_summary['best_bitstring']}]")
print(f"Best Energy Found:    {q_summary['best_energy']:.2f}")
print(f"Appearance Frequency: {q_summary['best_appearance_count']}/1000 ({q_summary['best_frequency_pct']}%) | Combined Twins: {comparison['combined_ground_states_frequency']}")
print(f"Exact Optimal Match:  {comparison['is_optimal_found']}")
print(f"Verdict:              {comparison['comparison_verdict']}")

## Part d: Parallel Computing (multiprocessing.Pool) [01 Mark, CO4]
- Benchmark independent restarts across 1, 2, and 4 worker processes.
- Compute Speed-up: $S_p = T_1 / T_p$.
- Compute Efficiency: $E_p = \frac{S_p}{p} \times 100\%$.

In [ ]:
parallel_results = run_parallel_restarts(
    Q, offset, worker_counts=[1, 2, 4], num_restarts=16, reads_per_restart=40, sweeps_per_restart=200
)
pd.DataFrame(parallel_results["scaling_table"])

## Part e: Conclusion in 3 Lines [0.5 Mark, CO4]
Strict 3-line synthesis of optimal decision choice and academic takeaways.

In [ ]:
conclusion_str = generate_three_line_conclusion(
    best_slot_1=q_summary["decoded"]["slot_1"],
    best_slot_2=q_summary["decoded"]["slot_2"],
    min_clash=q_summary["best_energy"],
    max_speedup=parallel_results["scaling_table"][-1]["speedup"],
    workers=4
)
print(conclusion_str)